In [7]:
import importlib
import torch
import model_def
importlib.reload(model_def)
from torch import nn
from pathlib import Path
from model_def import get_model
from test_mnist import download, load_images, load_labels

In [8]:
def load_mnist_train(root="./data"):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    image_file = "train-images-idx3-ubyte.gz"
    label_file = "train-labels-idx1-ubyte.gz"
    image_path = root / image_file
    label_path = root / label_file

    mnist_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"
    download(mnist_url + image_file, image_path)
    download(mnist_url + label_file, label_path)

    images = load_images(image_path)
    labels = load_labels(label_path)

    images = images[:, None, :, :]
    images = images.repeat(3, axis=1)

    return images, labels

In [5]:
print(get_model())

SimpleModel(
  (nnet): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1568, out_features=10, bias=True)
    (8): Softmax(dim=1)
  )
)


In [6]:
def train():
    images, labels = load_mnist_train()
    X = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    model = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    batch_size = 64
    n = X.shape[0]

    for epoch in range(10):
        perm = torch.randperm(n)          # shuffle indices each epoch
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = X[idx], y[idx]

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.state_dict(), "trained.pt")
    print("saved trained.pt")
train()


KeyboardInterrupt



In [11]:
#let's add color
def random_tint(images):
    n = images.shape[0]
    fg_color = torch.rand(n, 3, 1, 1)*0.8+0.2
    bg_color = torch.rand(n, 3, 1, 1)*0.8+0.2
    mask = images[ :, 0:1, :, :]
    colored_images = mask*fg_color + (1-mask)*bg_color
    return colored_images

In [12]:
def train_color():
    images, labels = load_mnist_train()
    X = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    model = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    batch_size = 64
    n = X.shape[0]
    for epoch in range(10):
        perm = torch.randperm(n)
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = X[idx], y[idx]

            xb = random_tint(xb)

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.state_dict(), "trained.pt")
    print("saved trained.pt")
train_color()

epoch 0, avg loss 1.9604
epoch 1, avg loss 1.8202
epoch 2, avg loss 1.8035
epoch 3, avg loss 1.7929
epoch 4, avg loss 1.7787
epoch 5, avg loss 1.6969
epoch 6, avg loss 1.6879
epoch 7, avg loss 1.6837
epoch 8, avg loss 1.6808
epoch 9, avg loss 1.6788
saved trained.pt
